# IS 445 Final Project — Part 1: Global energy and economic development

**Group member:** Naman Behl (individual project, Parts 1–3)

Working research question: How do energy use and energy intensity differ across countries and over time?

Draft exploration notebook: run each cell, inspect the results, and complete the personal observations before submission.

## Dataset provenance and access
**Name:** Our World in Data Energy dataset, release 2026-09-10.

OWID compiles and harmonizes country/year data from the Energy Institute Statistical Review of World Energy, Ember, the US Energy Information Administration, historical reconstructions, population sources and the Maddison Project Database. Each row represents a location (country or aggregate region) in a year, not an individual person. The indicators describe energy supply, electricity generation, fuel mix, economic output and population.

[Official documentation](https://catalog.ourworldindata.org/energy/owid_energy/) | [Fixed release CSV](https://catalog.ourworldindata.org/garden/energy/2026-09-10/owid_energy/owid_energy.csv) | [Codebook](https://catalog.ourworldindata.org/energy/owid_energy/owid_energy.codebook.csv)

**License:** OWID's own work is generally CC BY 4.0 (reuse and adaptation with attribution), but the catalog explicitly states that underlying third-party data retain the original providers' terms. For future public plots I will credit OWID and the specific underlying sources and review their terms before redistribution. Public access alone is not an unrestricted license. [OWID license explanation](https://github.com/owid/energy-data#license).

In [ ]:
import io
from urllib.request import urlopen
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://catalog.ourworldindata.org/garden/energy/2026-09-10/owid_energy/owid_energy.csv"
with urlopen(DATA_URL, timeout=90) as response:
    raw = response.read()
df = pd.read_csv(io.BytesIO(raw))
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]:,}")
print(f"CSV size: {len(raw)/1024**2:.2f} MiB")
print(f"DataFrame memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MiB")
df.head()

## Dataset size and hosting plan
The catalog lists 19,494 rows, 130 columns and approximately 8.7 MB of CSV data. The preceding cell measures the actual downloaded file and deep pandas memory use. This is below GitHub's 100 MiB per-file limit. I will load the fixed release directly from its source rather than duplicate it in GitHub. For future visualizations I can export smaller aggregated tables if needed. [GitHub size limits](https://docs.github.com/en/repositories/working-with-files/managing-large-files/about-large-files-on-github).

In [ ]:
df.info()
print("Location count:", df["country"].nunique())
print("Year range:", df["year"].min(), df["year"].max())
print("Duplicate location-year rows:", df.duplicated(["country","year"]).sum())
display(df.columns.to_list())

## Fields and data types
Country is nominal categorical text, year is temporal (stored as an integer in the CSV), and energy and economic measures are quantitative numeric fields. The following inventory names every field, its inferred storage type, nonmissing values and missing percentage. Missing values are not necessarily zero; coverage differs between indicators and years.

In [ ]:
inventory = pd.DataFrame({"dtype": df.dtypes.astype(str), "nonmissing": df.notna().sum(), "missing_pct": df.isna().mean().mul(100).round(2)})
display(inventory)
display(df.select_dtypes("number").describe().T)

In [ ]:
metrics = ["total_energy_twh", "total_energy_per_gdp_kwh_per_dollar"]
print(df[metrics].isna().sum())
display(df.groupby("year")[metrics].count().tail(20))

## Exploratory comparisons
I will start with a fixed selection of countries to avoid mixing regional aggregates with individual countries. Total energy supply (TWh) measures overall scale, while energy per unit of GDP (kWh per international dollar) measures energy intensity. The latter is not a direct measure of emissions or efficiency alone.

In [ ]:
countries = ["United States", "India", "China", "Germany", "France", "Japan", "United Kingdom"]
selected = df[df["country"].isin(countries) & df["year"].between(1990, 2022)].copy()
print("Available countries:", selected["country"].unique())
display(selected.groupby("country")[metrics].agg(["count","median","min","max"]))
selected.pivot_table(index="year", columns="country", values=metrics[0]).plot(figsize=(10,4), title="Total energy supply by year")
plt.ylabel("TWh")
plt.show()

## Main plot: Energy intensity over time
This plot compares the energy required per unit of economic output for selected countries over time. It is interesting because countries differ in economic structure, energy supply and development, so the trajectories may reveal changes that total energy consumption alone would obscure. The comparison is descriptive and does not establish causality.

In [ ]:
wide = selected.pivot_table(index="year", columns="country", values=metrics[1])
ax = wide.plot(figsize=(10,5), title="Energy intensity across selected countries")
ax.set_ylabel("kWh per international dollar of GDP")
ax.set_xlabel("Year")
plt.show()
display(wide.tail())

## Further exploration and debugging notes (complete after running)
Record the actual row count and memory printed above, observed trends in each plot, and at least one data limitation that you discovered. Keep any genuinely unsuccessful experiments here as commented-out code with the actual error or reason for changing approach. Do not invent failed attempts. Consider exploring missing values by year and comparing the results when you change the country selection.

**Next parts:** Consider a linked country selector, energy intensity time series and an energy mix comparison, all from this same fixed data release.